# Colab 16 — Pure 3-bin classification head + AdaptiveAvgPool(K) encoder

## What's new vs colab15

Two orthogonal changes:

1. **Encoder gains `AdaptiveAvgPool1d(K)`** before the flatten. The colab14/15 encoder
   was position-rigid: a small N-terminal insertion shifted all downstream features into
   different FC slots (the `4oo1I01` finding documented in `memory/architecture_insights.md`).
   The pool coarsens seq_len from 200 → K so small shifts get absorbed inside one bucket.

2. **Head becomes a 3-bin classifier** trained with plain cross-entropy. colab15 used
   band-weighted MSE on continuous normLev, which compresses predictions toward the
   training-label mean. Pure CE removes that pressure — the loss only wants the correct
   bin's logit higher than the other two.

## Ablation

K-value is the only architectural hyperparameter that's actually new. We run three values
side-by-side (K=8, K=16, K=32) with identical training data, seed, and eval — so any
differences are attributable to K alone.

## Deployment metrics (all three computed for k-NN)

The training objective is bin classification, but k-NN retrieval needs a continuous
score. We report three at inference time:

- `‖e_a − e_b‖` — continuous L2 in the embedding space. Direct comparable to colab15.
  Answers: "given a query, can we retrieve its high-sim partner?"
- `P(high) = softmax(logits)[2]` — the trained classifier's confidence. Answers: "is
  this pair in the high bin?"
- `E[bin midpoint] = P(far)·0.15 + P(mid)·0.50 + P(high)·0.85` — smoothed continuous
  blend.

## Eval set (frozen)

`sampledata/cath/cath_eval.csv.gz` — same as colab15: 5 high (4 valid + 1 short-rescue)
+ 2,945 mid + 2,000 far (seed=42 sample from 23K). Protein pool combines train70+test30
(~10,117 incl. 2 rescued short).

## Training data (unchanged)

Artificial AA pairs from `make_targeted_perturbation_pairs` — identical to colab14/15.
The alphabet-entropy floor caps achievable normLev at ~0.28, so the "far" bin is almost
empty in training. This is accepted as a no-op class for now; broadening training is
parked as a colab17/18 lever.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_eval.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn umap-learn --quiet


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128

# Band thresholds (unchanged from colab14/15)
BAND_LOW  = 0.30
BAND_HIGH = 0.70

# Bin index convention used everywhere
# 0 = far (<0.30), 1 = mid [0.30, 0.70), 2 = high (>=0.70)
BIN_NAMES = ['far', 'mid', 'high']
BIN_MIDPOINTS = np.array([0.15, 0.50, 0.85])

# K-ablation: three encoders trained side-by-side
K_VALUES = [8, 16, 32]

print(f'Bands: far<{BAND_LOW}, mid in [{BAND_LOW},{BAND_HIGH}), high>={BAND_HIGH}')
print(f'K ablation: {K_VALUES}')

## 3. Helpers — Levenshtein, encode, perturb, bin labeling

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def fast_norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_aa_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def bin_idx(x):
    if x < BAND_LOW:  return 0
    if x < BAND_HIGH: return 1
    return 2

def bin_indices_arr(xs):
    xs = np.asarray(xs)
    out = np.ones(xs.shape, dtype=np.int64)   # default = mid
    out[xs < BAND_LOW]  = 0
    out[xs >= BAND_HIGH] = 2
    return out

def bin_names_arr(xs):
    return np.array(BIN_NAMES)[bin_indices_arr(xs)]

## 4. Build training pairs — single-procedure, target-uniform sampling

Unchanged from colab13/14/15. Sample `t ~ Uniform[0,1]`, pick `k = round((1-t)*L)`,
apply `perturb`, use the actual computed `normLev` as the label. The label is then
binned for CE supervision; the continuous value is discarded after binning.

In [ ]:
def make_targeted_perturbation_pairs(seed_fn, alphabet, n_pairs):
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn()
        if seed is None or len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(0.0, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet)
        if 1 <= len(other) <= MAX_LEN:
            label = norm_lev(seed, other)
            pairs.append((seed, other, label))
    return pairs

print('Building training pairs (target-uniform random-AA perturbations)…')
train_pairs = make_targeted_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_TRAIN)
print(f'  got {len(train_pairs)} training pairs')

labels = np.array([p[2] for p in train_pairs])
bin_arr = bin_names_arr(labels)
bin_counts = {b: int((bin_arr == b).sum()) for b in BIN_NAMES}
print(f'  label range: [{labels.min():.3f}, {labels.max():.3f}]')
print(f'  bin counts:  {bin_counts}')

plt.figure(figsize=(8, 3))
plt.hist(labels, bins=40, edgecolor='k', alpha=0.75)
plt.axvline(BAND_LOW,  color='r', ls='--', label=f'far/mid ({BAND_LOW})')
plt.axvline(BAND_HIGH, color='g', ls='--', label=f'mid/high ({BAND_HIGH})')
plt.xlabel('normLev'); plt.ylabel('Count')
plt.title('Training-pair label distribution + bin thresholds')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Architecture — encoder (with AdaptiveAvgPool) + classification head

Encoder is parameterized by `K` (the pool's output dim). The FC input becomes
`hidden2 * K` instead of `hidden2 * MAX_LEN`. A 4-character N-term shift on
MAX_LEN=200 corresponds to 4/(200/K) of a bucket — for K=16 that's 0.32 of a bucket,
mostly absorbed.

Head is `Linear(128, 64) → LeakyReLU(0.01) → Linear(64, 3)`. The classifier eats
`|e_a − e_b|` (symmetric, since `lev(a,b) = lev(b,a)`). LeakyReLU instead of ReLU
to keep dying-neuron risk near zero.

The encoder is reusable standalone: `model.encoder(x)` produces a 128-d L2-normalized
embedding for k-NN retrieval via `‖e_a − e_b‖`.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)        # (B, embed_dim, L)
        h = F.relu(self.conv1(e))
        h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)                      # zero out PAD activations
        h = self.pool(h)                               # (B, hidden2, K)
        h = h.flatten(1)                               # (B, hidden2 * K)
        z = self.fc(h)
        return F.normalize(z, p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins),
        )

    def forward(self, a, b):
        e_a = self.encoder(a)
        e_b = self.encoder(b)
        diff = torch.abs(e_a - e_b)
        return self.head(diff)                         # raw logits, (B, 3)

    def encode(self, x):
        return self.encoder(x)


# Sanity: param counts per K
for K in K_VALUES:
    m = SiameseClassifier(K)
    n = sum(p.numel() for p in m.parameters())
    fc_w = m.encoder.fc.weight.numel()
    print(f'K={K:>2}: total params = {n:>9,}    FC weights = {fc_w:>7,}')

## 6. Dataset — yields (a_idx, b_idx, bin_label_int)

In [ ]:
class PairDatasetCE(Dataset):
    def __init__(self, pairs):
        # Pre-encode and pre-bin once for speed
        self.a_encoded = [encode_pad(a) for a, b, _ in pairs]
        self.b_encoded = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx(label) for _, _, label in pairs],
                                 dtype=torch.long)

    def __len__(self): return len(self.bins)
    def __getitem__(self, i):
        return self.a_encoded[i], self.b_encoded[i], self.bins[i]

train_ds = PairDatasetCE(train_pairs)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f'Train dataset: {len(train_ds)} pairs')

## 7. Training function

`train_one(K)` returns `(trained_model, losses)`. Called once per K value.
Plain `CrossEntropyLoss()` (no class weights — see `memory/next_iteration_plan.md`
fork #5 resolution).

In [ ]:
def train_one(K, n_epochs=NUM_EPOCHS, seed=SEED, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = SiameseClassifier(K).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    losses = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        ep_loss = 0.0; nb = 0
        for a, b, bins in train_dl:
            a = a.to(device); b = b.to(device); bins = bins.to(device)
            logits = model(a, b)
            loss = criterion(logits, bins)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            ep_loss += loss.item(); nb += 1
        losses.append(ep_loss / nb)
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  K={K} epoch {epoch:3d}/{n_epochs} — CE: {losses[-1]:.4f}')
    return model, losses

## 8. Load eval set + protein pool

Same logic as colab15 — combined train70+test30 pool, standard-char + length-match
filter, rescued short proteins explicitly whitelisted.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')

prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()

RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool: {len(prot_df)} (incl. {len(prot_df) - in_range.sum()} rescued short)')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
print(f'Eval pairs: {len(eval_df)}')
print(eval_df['aa_bin'].value_counts().reindex(BIN_NAMES).to_string())

In [ ]:
def make_pairs(df, rep):
    score_col = 'aa_score' if rep == 'aa' else 'ss_score'
    lookup = id_to_aa if rep == 'aa' else id_to_ss
    out = []
    for _, r in df.iterrows():
        a_id, b_id = r['domain_a'], r['domain_b']
        if a_id not in lookup or b_id not in lookup: continue
        out.append((lookup[a_id], lookup[b_id], float(r[score_col])))
    return out

eval_pairs_aa = make_pairs(eval_df, 'aa')
eval_pairs_ss = make_pairs(eval_df, 'ss')
print(f'AA eval pairs: {len(eval_pairs_aa)}    SS eval pairs: {len(eval_pairs_ss)}')

## 9. Predict helpers — three deployment scores per pair

For each eval pair (and later for retrieval), compute:
- `sim_l2 = 1 − ‖e_a − e_b‖ / 2` (encoder-only, no head)
- `p_high = softmax(logits)[2]`
- `emid = sum(softmax · BIN_MIDPOINTS)`
- `argmax_bin` (the classifier's hard prediction)

In [ ]:
@torch.no_grad()
def predict_pairs(model, pairs, batch_size=512):
    model.eval()
    true_v, sim_l2, p_high, emid, argmax_bin = [], [], [], [], []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
        b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
        e_a = model.encoder(a); e_b = model.encoder(b)
        l2 = 1.0 - torch.linalg.vector_norm(e_a - e_b, ord=2, dim=1) / 2.0
        logits = model.head(torch.abs(e_a - e_b))
        probs = F.softmax(logits, dim=1).cpu().numpy()      # (B, 3)
        true_v.extend([p[2] for p in batch])
        sim_l2.extend(l2.cpu().numpy().tolist())
        p_high.extend(probs[:, 2].tolist())
        emid.extend((probs * BIN_MIDPOINTS[None, :]).sum(axis=1).tolist())
        argmax_bin.extend(probs.argmax(axis=1).tolist())
    return (np.array(true_v), np.array(sim_l2), np.array(p_high),
            np.array(emid), np.array(argmax_bin))

## 10. Banded metrics — per-bin r/MAE/bias + AUROC

In [ ]:
def per_band_bias(true_v, pred_v):
    tb = bin_names_arr(true_v)
    out = {}
    for b in BIN_NAMES:
        m = (tb == b)
        if m.sum() == 0:
            out[b] = dict(n=0, true_mean=float('nan'), pred_mean=float('nan'),
                          bias=float('nan'), pred_std=float('nan'))
            continue
        out[b] = dict(
            n=int(m.sum()),
            true_mean=float(true_v[m].mean()),
            pred_mean=float(pred_v[m].mean()),
            bias=float(pred_v[m].mean() - true_v[m].mean()),
            pred_std=float(pred_v[m].std()),
        )
    return out

def banded_metrics(name, true_v, pred_v, score_for_auroc=None):
    # `pred_v` is the continuous prediction used for per-bin r/MAE/bias.
    # `score_for_auroc` is the score for AUROC; defaults to pred_v.
    if score_for_auroc is None:
        score_for_auroc = pred_v
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    pr_all = pearsonr(true_v, pred_v)[0]
    sr_all = spearmanr(true_v, pred_v)[0]
    hi = true_v >= BAND_HIGH
    mi = (true_v >= BAND_LOW) & (true_v < BAND_HIGH)
    fa = true_v < BAND_LOW
    pr_hi = pearsonr(true_v[hi], pred_v[hi])[0] if hi.sum() > 10 else float('nan')
    pr_mi = pearsonr(true_v[mi], pred_v[mi])[0] if mi.sum() > 10 else float('nan')
    pr_fa = pearsonr(true_v[fa], pred_v[fa])[0] if fa.sum() > 10 else float('nan')
    mae_hi = np.abs(pred_v[hi] - true_v[hi]).mean() if hi.sum() > 0 else float('nan')
    mae_mi = np.abs(pred_v[mi] - true_v[mi]).mean() if mi.sum() > 0 else float('nan')
    mae_fa = np.abs(pred_v[fa] - true_v[fa]).mean() if fa.sum() > 0 else float('nan')
    y_hi = (true_v >= BAND_HIGH).astype(int)
    auroc = (roc_auc_score(y_hi, score_for_auroc)
             if 0 < y_hi.sum() < len(y_hi) else float('nan'))
    bias = per_band_bias(true_v, pred_v)
    print(f'--- {name} (n={len(true_v)}) ---')
    print(f'  Pearson r (all):  {pr_all:+.3f}    Spearman: {sr_all:+.3f}')
    print(f'  Per-bin r:  high={pr_hi:+.3f}  mid={pr_mi:+.3f}  far={pr_fa:+.3f}')
    print(f'  Per-bin MAE: high={mae_hi:.3f}  mid={mae_mi:.3f}  far={mae_fa:.3f}')
    print(f'  AUROC is-high: {auroc:.3f}')
    for b in BIN_NAMES[::-1]:
        v = bias[b]
        print(f'    {b:>4}: n={v["n"]:>5}  bias={v["bias"]:+.3f}  pred_std={v["pred_std"]:.3f}')
    return dict(pr_all=pr_all, sr_all=sr_all,
                pr_hi=pr_hi, pr_mi=pr_mi, pr_fa=pr_fa,
                mae_hi=mae_hi, mae_mi=mae_mi, mae_fa=mae_fa,
                auroc=auroc, bias=bias, n=len(true_v))

## 11. Retrieval — per metric (L2 / P(high) / E[bin midpoint])

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval()
    out = []
    for i in range(0, len(ids), batch_size):
        chunk = ids[i:i+batch_size]
        x = torch.stack([encode_pad(seq_lookup[d]) for d in chunk]).to(device)
        out.append(model.encoder(x))
    return torch.cat(out, 0)

@torch.no_grad()
def head_probs(model, e_query, pool_emb, batch_size=4096):
    # For one query (d,) and pool (N, d), return softmax probs (N, 3).
    N = pool_emb.shape[0]
    probs = torch.empty((N, 3), device=device)
    e_q = e_query.unsqueeze(0)
    for i in range(0, N, batch_size):
        diff = torch.abs(pool_emb[i:i+batch_size] - e_q)
        logits = model.head(diff)
        probs[i:i+batch_size] = F.softmax(logits, dim=1)
    return probs

HIGH_PAIRS = eval_df[eval_df['aa_bin'] == 'high'][['domain_a', 'domain_b']].values.tolist()
print('High-AA pairs for retrieval:')
for a, b in HIGH_PAIRS: print(f'  {a} ↔ {b}')

def run_retrieval(model, seq_lookup, label):
    # Returns dict with per-metric ranks + hits for the standard k-values.
    model.eval()
    pool_emb = encode_pool(model, seq_lookup, all_domains)
    id_to_idx = {d: i for i, d in enumerate(all_domains)}
    K_RECALL = (1, 5, 10, 50)
    METRIC_NAMES = ['L2', 'P(high)', 'E[bin_mid]']
    results = {m: {'ranks': [], 'hits': {k: 0 for k in K_RECALL}} for m in METRIC_NAMES}
    print(f'\n--- {label} retrieval (pool={len(all_domains)}) ---')
    print(f'{"query":<10} {"partner":<10}  {"rank_L2":>8} {"rank_Phigh":>11} {"rank_Emid":>10}')
    for a, b in HIGH_PAIRS:
        for q, partner in [(a, b), (b, a)]:
            if q not in id_to_idx or partner not in id_to_idx:
                continue
            qi, pi = id_to_idx[q], id_to_idx[partner]
            e_q = pool_emb[qi]
            # L2 score (higher = more similar)
            l2_sim = (1.0 - torch.linalg.vector_norm(pool_emb - e_q, ord=2, dim=1) / 2.0
                      ).cpu().numpy()
            l2_sim[qi] = -np.inf
            order_l2 = np.argsort(-l2_sim)
            rank_l2 = int(np.where(order_l2 == pi)[0][0]) + 1

            # Head scores
            probs = head_probs(model, e_q, pool_emb).cpu().numpy()
            p_hi = probs[:, 2]
            emid = (probs * BIN_MIDPOINTS[None, :]).sum(axis=1)
            p_hi[qi] = -np.inf; emid[qi] = -np.inf
            rank_phigh = int(np.where(np.argsort(-p_hi) == pi)[0][0]) + 1
            rank_emid  = int(np.where(np.argsort(-emid) == pi)[0][0]) + 1

            for m_name, r in [('L2', rank_l2), ('P(high)', rank_phigh), ('E[bin_mid]', rank_emid)]:
                results[m_name]['ranks'].append((q, partner, r))
                for k in K_RECALL:
                    if r <= k: results[m_name]['hits'][k] += 1
            print(f'{q:<10} {partner:<10}  {rank_l2:>8} {rank_phigh:>11} {rank_emid:>10}')

    print(f'\n  Hits@k summary:')
    print(f'  {"metric":<12} {"@1":>6} {"@5":>6} {"@10":>6} {"@50":>6}')
    for m in METRIC_NAMES:
        nq = len(results[m]['ranks'])
        h = results[m]['hits']
        print(f'  {m:<12} {h[1]:>3}/{nq:<2} {h[5]:>3}/{nq:<2} {h[10]:>3}/{nq:<2} {h[50]:>3}/{nq:<2}')
    return results

## 12. Eval block — all metrics for one trained model

In [ ]:
def evaluate(model, K):
    print(f'\n========== K = {K} eval ==========')
    # Predict on AA and SS eval pairs (3 continuous scores per pair)
    t_aa, l2_aa, phigh_aa, emid_aa, argmax_aa = predict_pairs(model, eval_pairs_aa)
    t_ss, l2_ss, phigh_ss, emid_ss, argmax_ss = predict_pairs(model, eval_pairs_ss)

    # Per-bin metrics — three deployment scores each, AA and SS
    print('\n  >> AA representation (label = aa_score)')
    m_aa_l2    = banded_metrics(f'AA L2-sim   (K={K})',         t_aa, l2_aa,    score_for_auroc=l2_aa)
    m_aa_phigh = banded_metrics(f'AA P(high)  (K={K})',         t_aa, phigh_aa, score_for_auroc=phigh_aa)
    m_aa_emid  = banded_metrics(f'AA E[bin_mid] (K={K})',       t_aa, emid_aa,  score_for_auroc=emid_aa)

    print('\n  >> SS representation (label = ss_score, cross-rep)')
    m_ss_l2    = banded_metrics(f'SS L2-sim   (K={K})',         t_ss, l2_ss,    score_for_auroc=l2_ss)
    m_ss_phigh = banded_metrics(f'SS P(high)  (K={K})',         t_ss, phigh_ss, score_for_auroc=phigh_ss)
    m_ss_emid  = banded_metrics(f'SS E[bin_mid] (K={K})',       t_ss, emid_ss,  score_for_auroc=emid_ss)

    # Classification confusion matrix — pure argmax of softmax vs true bin
    true_bin_aa = bin_indices_arr(t_aa)
    cm_aa = confusion_matrix(true_bin_aa, argmax_aa, labels=[0, 1, 2])
    print(f'\n  Confusion matrix AA (rows=true [far,mid,high], cols=pred):')
    print(pd.DataFrame(cm_aa, index=BIN_NAMES, columns=BIN_NAMES).to_string())

    true_bin_ss = bin_indices_arr(t_ss)
    cm_ss = confusion_matrix(true_bin_ss, argmax_ss, labels=[0, 1, 2])
    print(f'\n  Confusion matrix SS (rows=true [far,mid,high], cols=pred):')
    print(pd.DataFrame(cm_ss, index=BIN_NAMES, columns=BIN_NAMES).to_string())

    # Identity test
    from random import sample as rsample
    sample_ids = rsample(all_domains, 100)
    id_pairs_aa = [(id_to_aa[d], id_to_aa[d], 1.0) for d in sample_ids]
    _, ident_l2, ident_phigh, _, ident_bin = predict_pairs(model, id_pairs_aa)
    print(f'\n  Identity AA: L2-sim mean={ident_l2.mean():.4f}±{ident_l2.std():.4f}, '
          f'P(high) mean={ident_phigh.mean():.4f}, argmax=high in {(ident_bin==2).mean()*100:.0f}%')

    # k-NN retrieval — three metrics, AA and SS
    retr_aa = run_retrieval(model, id_to_aa, f'AA repr (K={K})')
    retr_ss = run_retrieval(model, id_to_ss, f'SS repr (K={K})')

    return {
        'K': K,
        'aa': {'true': t_aa, 'l2': l2_aa, 'phigh': phigh_aa, 'emid': emid_aa,
               'argmax': argmax_aa, 'metrics_l2': m_aa_l2, 'metrics_phigh': m_aa_phigh,
               'metrics_emid': m_aa_emid, 'cm': cm_aa},
        'ss': {'true': t_ss, 'l2': l2_ss, 'phigh': phigh_ss, 'emid': emid_ss,
               'argmax': argmax_ss, 'metrics_l2': m_ss_l2, 'metrics_phigh': m_ss_phigh,
               'metrics_emid': m_ss_emid, 'cm': cm_ss},
        'identity': {'l2': ident_l2, 'phigh': ident_phigh, 'bin': ident_bin},
        'retr_aa': retr_aa, 'retr_ss': retr_ss,
    }

## 13. Run the K-ablation — train and evaluate K ∈ {8, 16, 32}

Each K trains an independent encoder + head with the same seed and training data.
Three runs total. Each is faster than colab15 thanks to smaller FC.

In [ ]:
results = {}
training_losses = {}
for K in K_VALUES:
    print(f'\n############## Training K = {K} ##############')
    model_K, losses_K = train_one(K)
    training_losses[K] = losses_K
    results[K] = evaluate(model_K, K)
    # Keep model around for downstream plots
    results[K]['model'] = model_K

# Compare training loss curves
fig, ax = plt.subplots(figsize=(8, 4))
for K in K_VALUES:
    ax.plot(training_losses[K], label=f'K={K}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training loss — three K values')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 14. Comparison table across K — and colab15 baseline

Headline numbers to copy into BENCHMARKS.md. The colab15 reference column is
filled in by hand from `memory/colab_iters_summary.md` so we can read movement
at a glance.

In [ ]:
COLAB15_REF = {
    'aa_auroc': 0.911, 'aa_mae_hi': 0.154, 'aa_hits10': '6/10',
    'ss_auroc': 0.935, 'ss_mae_hi': 0.063, 'ss_hits10': '2/10',
}

def metric_for(res, side, score):
    # Pick the right metric dict given side ('aa'/'ss') and score ('l2'/'phigh'/'emid').
    key = {'l2': 'metrics_l2', 'phigh': 'metrics_phigh', 'emid': 'metrics_emid'}[score]
    return res[side][key]

def hits10_for(res, side, score):
    key = {'l2': 'L2', 'phigh': 'P(high)', 'emid': 'E[bin_mid]'}[score]
    retr = res['retr_aa' if side == 'aa' else 'retr_ss'][key]
    return f"{retr['hits'][10]}/{len(retr['ranks'])}"

print('\n==================== HEADLINE TABLE ====================')
print('Score = which deployment metric was used for the row.')
print('hits@10 reported for the corresponding metric; AUROC + MAE(hi) use the same.')
print()
header = f'{"K":>3} {"side":>4} {"score":>10}  {"AUROC":>6} {"MAE(hi)":>7} {"hits@10":>8}  ' \
         f'{"bias(hi)":>8} {"bias(mid)":>9} {"bias(far)":>9}'
print(header)
print('-' * len(header))
for K in K_VALUES:
    for side in ['aa', 'ss']:
        for score in ['l2', 'phigh', 'emid']:
            m = metric_for(results[K], side, score)
            h10 = hits10_for(results[K], side, score)
            print(f'{K:>3} {side:>4} {score:>10}  '
                  f'{m["auroc"]:>6.3f} {m["mae_hi"]:>7.3f} {h10:>8}  '
                  f'{m["bias"]["high"]["bias"]:>+8.3f} '
                  f'{m["bias"]["mid"]["bias"]:>+9.3f} '
                  f'{m["bias"]["far"]["bias"]:>+9.3f}')
    print()
print(f'colab15 ref AA: AUROC={COLAB15_REF["aa_auroc"]}  MAE(hi)={COLAB15_REF["aa_mae_hi"]}  hits@10={COLAB15_REF["aa_hits10"]}')
print(f'colab15 ref SS: AUROC={COLAB15_REF["ss_auroc"]}  MAE(hi)={COLAB15_REF["ss_mae_hi"]}  hits@10={COLAB15_REF["ss_hits10"]}')

## 15. Per-pair retrieval ranks — best K, all three metrics

Like the colab15 per-pair table, but with three rank columns (one per deployment
metric). Lets us see which metric is best per query and whether the new
architecture rescues the `4oo1I01` outlier.

In [ ]:
def best_K_by_aa_hits10_l2(results):
    return max(K_VALUES, key=lambda K: results[K]['retr_aa']['L2']['hits'][10])

best_K = best_K_by_aa_hits10_l2(results)
print(f'Best K by AA-L2 hits@10: K={best_K}')

print('\nPer-pair AA retrieval ranks (best K):')
print(f'{"query":<10} {"partner":<10} {"rank_L2":>8} {"rank_Phigh":>11} {"rank_Emid":>10}')
ranks_l2    = {(q, p): r for q, p, r in results[best_K]['retr_aa']['L2']['ranks']}
ranks_phigh = {(q, p): r for q, p, r in results[best_K]['retr_aa']['P(high)']['ranks']}
ranks_emid  = {(q, p): r for q, p, r in results[best_K]['retr_aa']['E[bin_mid]']['ranks']}
for (q, p) in ranks_l2.keys():
    print(f'{q:<10} {p:<10} {ranks_l2[(q,p)]:>8} {ranks_phigh[(q,p)]:>11} {ranks_emid[(q,p)]:>10}')

print('\nPer-pair SS retrieval ranks (best K):')
print(f'{"query":<10} {"partner":<10} {"rank_L2":>8} {"rank_Phigh":>11} {"rank_Emid":>10}')
ranks_l2_ss    = {(q, p): r for q, p, r in results[best_K]['retr_ss']['L2']['ranks']}
ranks_phigh_ss = {(q, p): r for q, p, r in results[best_K]['retr_ss']['P(high)']['ranks']}
ranks_emid_ss  = {(q, p): r for q, p, r in results[best_K]['retr_ss']['E[bin_mid]']['ranks']}
for (q, p) in ranks_l2_ss.keys():
    print(f'{q:<10} {p:<10} {ranks_l2_ss[(q,p)]:>8} {ranks_phigh_ss[(q,p)]:>11} {ranks_emid_ss[(q,p)]:>10}')

## 16. Prediction-bias figure — best K, AA vs SS

In [ ]:
def bias_for_plot(m, side):
    return [results[best_K][side][m]['bias'][b]['bias'] for b in BIN_NAMES[::-1]]
def std_for_plot(m, side):
    return [results[best_K][side][m]['bias'][b]['pred_std'] for b in BIN_NAMES[::-1]]

# Use L2 score since that's the colab15-comparable continuous metric
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(3); w = 0.35
labels_bin = BIN_NAMES[::-1]  # high, mid, far

aa_bias = [results[best_K]['aa']['metrics_l2']['bias'][b]['bias'] for b in labels_bin]
ss_bias = [results[best_K]['ss']['metrics_l2']['bias'][b]['bias'] for b in labels_bin]
aa_std  = [results[best_K]['aa']['metrics_l2']['bias'][b]['pred_std'] for b in labels_bin]
ss_std  = [results[best_K]['ss']['metrics_l2']['bias'][b]['pred_std'] for b in labels_bin]

axL.bar(x - w/2, aa_bias, w, label='AA', color='tab:blue')
axL.bar(x + w/2, ss_bias, w, label='SS', color='tab:green')
axL.axhline(0, color='k', lw=0.5)
axL.set_xticks(x); axL.set_xticklabels(labels_bin)
axL.set_ylabel('mean(pred) − mean(true)')
axL.set_title(f'Per-bin bias (K={best_K}, L2-sim score)')
axL.legend(); axL.grid(True, alpha=0.3)

axR.bar(x - w/2, aa_std, w, label='AA', color='tab:blue')
axR.bar(x + w/2, ss_std, w, label='SS', color='tab:green')
axR.set_xticks(x); axR.set_xticklabels(labels_bin)
axR.set_ylabel('std(pred)')
axR.set_title(f'Per-bin prediction spread (K={best_K})')
axR.legend(); axR.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 17. Scatter — predicted vs true (best K, L2-sim, AA + SS)

Direct visual comparable to colab15 sections 10/11. If compression is broken,
predictions should span more of the y-axis instead of clustering in [0.4, 0.65].

In [ ]:
colors_bin = {'far': 'tab:gray', 'mid': 'tab:orange', 'high': 'tab:red'}

def scatter_eval(side, score_key, ax, title):
    t = results[best_K][side]['true']
    p = results[best_K][side][score_key]
    tb = bin_names_arr(t)
    for b in BIN_NAMES:
        m = tb == b
        ax.scatter(t[m], p[m], s=12 if b != 'high' else 80,
                   alpha=0.35 if b != 'high' else 1.0, color=colors_bin[b],
                   edgecolors='black' if b == 'high' else 'none', linewidths=0.6,
                   label=f'{b} (n={m.sum()})')
    ax.plot([0, 1], [0, 1], 'r-', lw=1.5, label='y = x')
    for v in (BAND_LOW, BAND_HIGH):
        ax.axhline(v, color='k', ls=':', lw=0.5); ax.axvline(v, color='k', ls=':', lw=0.5)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('True normLev'); ax.set_ylabel('L2-derived predicted sim')
    ax.set_title(title); ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.3)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 6))
scatter_eval('aa', 'l2', axL, f'AA — K={best_K} (Pearson r = {results[best_K]["aa"]["metrics_l2"]["pr_all"]:+.3f})')
scatter_eval('ss', 'l2', axR, f'SS — K={best_K} (Pearson r = {results[best_K]["ss"]["metrics_l2"]["pr_all"]:+.3f})')
plt.tight_layout(); plt.show()

## 18. Final summary — what to paste into BENCHMARKS.md

Compact table for the colab16 row(s) in BENCHMARKS.md. Three rows per K (one per
deployment metric × representation) or pick the best metric per K.

In [ ]:
print('=' * 80)
print('COLAB16 SUMMARY — for BENCHMARKS.md')
print('=' * 80)
print(f'Best K (by AA-L2 hits@10): K = {best_K}')
print()
print('AA representation:')
print(f'  {"score":<10}  {"AUROC":>6} {"MAE(hi)":>7} {"hits@10":>8} {"bias(hi)":>8}')
for sc in ['l2', 'phigh', 'emid']:
    m = metric_for(results[best_K], 'aa', sc)
    h10 = hits10_for(results[best_K], 'aa', sc)
    print(f'  {sc:<10}  {m["auroc"]:>6.3f} {m["mae_hi"]:>7.3f} {h10:>8} {m["bias"]["high"]["bias"]:>+8.3f}')
print()
print('SS representation (cross-rep):')
print(f'  {"score":<10}  {"AUROC":>6} {"MAE(hi)":>7} {"hits@10":>8} {"bias(hi)":>8}')
for sc in ['l2', 'phigh', 'emid']:
    m = metric_for(results[best_K], 'ss', sc)
    h10 = hits10_for(results[best_K], 'ss', sc)
    print(f'  {sc:<10}  {m["auroc"]:>6.3f} {m["mae_hi"]:>7.3f} {h10:>8} {m["bias"]["high"]["bias"]:>+8.3f}')
print()
print('All K hits@10 (L2 score):')
for K in K_VALUES:
    aa_h = results[K]["retr_aa"]["L2"]["hits"][10]
    ss_h = results[K]["retr_ss"]["L2"]["hits"][10]
    nq_aa = len(results[K]["retr_aa"]["L2"]["ranks"])
    nq_ss = len(results[K]["retr_ss"]["L2"]["ranks"])
    print(f'  K={K:>2}:  AA = {aa_h}/{nq_aa}    SS = {ss_h}/{nq_ss}')

## 19. Wall-clock benchmark — Levenshtein DP vs trained NN (per query)

**Status: NOT YET RUN.** The whole thesis premise is that the NN is *faster* than
the O(n·m) Levenshtein DP at retrieval time. Without a wall-clock benchmark, the
speedup claim is theoretical. Run this cell **once per architecture change** to
get a single defensible number for the writeup. It is independent of the per-K
eval above — pure timing, no metric changes.

What it measures:
- **Lev cost per query:** rapidfuzz C-backed `Levenshtein.distance` against the full pool.
- **NN cost per query:** one encoder forward pass on the query + vectorized L2
  distance against the **precomputed** pool embeddings. Pool encoding is a
  one-time amortized cost (reported separately).
- **Extrapolation:** linear scaling to 1M / UniRef50 (~50M) / BFD (~2.5B). This
  is a lower bound for Lev (real DBs have longer sequences than CATH-S20) and
  conservative for NN (FAISS/HNSW gives sublinear retrieval).

Why it isn't in the per-K loop: timing is sensitive to other processes on the
Colab runtime, so we want a clean separate run, not three timings interleaved
with training. Pick the headline K (K=16 currently) and benchmark only that one.

In [ ]:
# ============================================================================
# Wall-clock: Levenshtein DP vs trained NN, per query (10 high-AA queries)
# RUN-ONCE benchmark — not part of the per-K eval.
# ============================================================================
import time
from rapidfuzz.distance import Levenshtein as RFLev

BENCH_K = best_K_by_aa_hits10_l2(results)   # headline K (e.g. 16)
bench_model = results[BENCH_K]['model']
bench_model.eval()
print(f'Benchmarking K = {BENCH_K} on device = {device}')
print(f'Pool size = {len(all_domains)}    (mean len = {int(prot_df["aa_len"].mean())})')

# --- NN one-time cost: encode full pool ---
if device.type == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
pool_emb_bench = encode_pool(bench_model, id_to_aa, all_domains)
if device.type == 'cuda': torch.cuda.synchronize()
nn_pool_encode_sec = time.perf_counter() - t0
print(f'NN pool-encode (amortized one-time): {nn_pool_encode_sec*1000:.1f} ms')

# --- Build the 10-query list (matches the retrieval eval) ---
bench_queries = [(q, partner)
                 for a, b in HIGH_PAIRS
                 for q, partner in [(a, b), (b, a)]
                 if q in id_to_aa and partner in id_to_aa]
print(f'Bench queries: {len(bench_queries)} (5 high-AA pairs x 2 directions)')

# --- Per-query timing ---
pool_seqs_aa = [id_to_aa[d] for d in all_domains]
lev_times = []
nn_times = []
for q, _ in bench_queries:
    q_seq = id_to_aa[q]

    # Lev: full sweep over pool (C-backed)
    t0 = time.perf_counter()
    _ = [RFLev.distance(q_seq, s) for s in pool_seqs_aa]
    lev_times.append(time.perf_counter() - t0)

    # NN: one encoder forward + vectorized L2 against precomputed pool
    q_tensor = encode_pad(q_seq).unsqueeze(0).to(device)
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        e_q = bench_model.encoder(q_tensor).squeeze(0)
        _ = torch.linalg.vector_norm(pool_emb_bench - e_q, ord=2, dim=1)
    if device.type == 'cuda': torch.cuda.synchronize()
    nn_times.append(time.perf_counter() - t0)

lev_times = np.array(lev_times)
nn_times = np.array(nn_times)

# --- Report ---
print('\n' + '=' * 72)
print(f'Wall-clock per query  (pool = {len(all_domains)})')
print('=' * 72)
print(f'  Levenshtein (rapidfuzz, C-backed): {lev_times.mean()*1000:>8.1f} +/- {lev_times.std()*1000:.1f} ms')
print(f'  NN (1 fwd + L2 vs pool):           {nn_times.mean()*1000:>8.1f} +/- {nn_times.std()*1000:.1f} ms')
print(f'  Speedup:                           ~{(lev_times.mean()/nn_times.mean()):.0f}x')

# --- Extrapolation (linear in pool size) ---
def fmt(t):
    if t < 1: return f'{t*1000:.0f} ms'
    if t < 60: return f'{t:.1f} sec'
    if t < 3600: return f'{t/60:.1f} min'
    if t < 86400: return f'{t/3600:.1f} hr'
    return f'{t/86400:.1f} days'

print('\nLinear extrapolation to larger pools:')
print(f'  {"pool":<18} {"Lev / query":>14} {"NN / query":>14}')
for pool_size, label in [(len(all_domains),  'this eval'),
                         (1_000_000,         '1M'),
                         (50_000_000,        'UniRef50 ~50M'),
                         (2_500_000_000,     'BFD ~2.5B')]:
    factor = pool_size / len(all_domains)
    print(f'  {label:<18} {fmt(lev_times.mean()*factor):>14} {fmt(nn_times.mean()*factor):>14}')

print('\nFootnotes:')
print('  - Lev = rapidfuzz.distance.Levenshtein.distance (C-backed, fastest stdlib option)')
print(f'  - NN  = encoder fwd pass + torch.linalg.vector_norm on {device.type.upper()}')
print('  - Pool encode is one-time amortized over many queries; excluded from per-query cost')
print('  - Linear scaling is a *lower bound* for Lev (UniRef50/BFD have longer seqs than CATH-S20)')
print('  - NN scales sublinearly with FAISS/HNSW index in production; extrapolation is')
print('  - NN scales sublinearly with FAISS/HNSW index in production; extrapolation is')
print('    therefore conservative for NN (real speedup at scale will be larger)')


## 20. Embedding-space visualisations

Two figures to make the "the encoder learned a similarity manifold" claim visible:

**Figure 1 — Pool embeddings (2D UMAP).** All ~10K protein embeddings projected
to 2D. Points colored by protein length. The 5 high-AA partner pairs are drawn
as red line segments. Short red lines = encoder places true partners close
together in embedding space, which is exactly what k-NN retrieval needs.

**Figure 2 — Pair-difference vectors |e_a − e_b| (2D PCA).** One point per AA
eval pair, colored by true normLev band (far/mid/high). This is the colab15
figure redone under CE training. Under colab15's MSE training PC1+PC2 only
explained ~17% variance with no band structure. If colab16's discrete-band
geometry shows up, you should see three colored clusters instead of a blob.

In [ ]:
import umap
from sklearn.decomposition import PCA

VIZ_K = best_K
viz_model = results[VIZ_K]['model']
viz_model.eval()
print(f'Visualising K = {VIZ_K}')

# ----------------------------------------------------------------------------
# Figure 1 — 2D UMAP of pool embeddings, with high-AA partner pairs overlaid
# ----------------------------------------------------------------------------
pool_emb_viz = encode_pool(viz_model, id_to_aa, all_domains).cpu().numpy()
print(f'Pool embeddings shape: {pool_emb_viz.shape}')

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
pool_2d = reducer.fit_transform(pool_emb_viz)
id_to_xy = {d: pool_2d[i] for i, d in enumerate(all_domains)}
lens = np.array([len(id_to_aa[d]) for d in all_domains])

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(pool_2d[:, 0], pool_2d[:, 1], c=lens, cmap='viridis',
                s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Protein length (AA)')

# Overlay the 5 high-AA pairs as red line segments
for a, b in HIGH_PAIRS:
    if a in id_to_xy and b in id_to_xy:
        xa, ya = id_to_xy[a]
        xb, yb = id_to_xy[b]
        ax.plot([xa, xb], [ya, yb], color='red', lw=1.5, alpha=0.9, zorder=5)
        ax.scatter([xa, xb], [ya, yb], color='red', s=80,
                   edgecolors='black', linewidths=1.2, zorder=6)
        ax.annotate(f'{a[:6]}\u2194{b[:6]}',
                    ((xa + xb) / 2, (ya + yb) / 2),
                    fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white',
                              ec='red', alpha=0.8))

ax.set_title(f'Pool embeddings (UMAP 2D) - K={VIZ_K}\n'
             f'red segments connect the 5 high-AA partner pairs')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ----------------------------------------------------------------------------
# Figure 2 - 2D PCA of pair-difference vectors |e_a - e_b| (AA eval pairs)
# ----------------------------------------------------------------------------
@torch.no_grad()
def get_pair_diffs(model, pair_list, batch_size=512):
    model.eval()
    diffs, scores = [], []
    for i in range(0, len(pair_list), batch_size):
        batch = pair_list[i:i+batch_size]
        a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
        b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
        e_a = model.encoder(a); e_b = model.encoder(b)
        diffs.append(torch.abs(e_a - e_b).cpu().numpy())
        scores.extend([p[2] for p in batch])
    return np.vstack(diffs), np.array(scores)

diffs_aa, scores_aa = get_pair_diffs(viz_model, eval_pairs_aa)
print(f'Pair-diff vectors shape: {diffs_aa.shape}')

pca = PCA(n_components=2, random_state=SEED)
diffs_2d = pca.fit_transform(diffs_aa)
pc1_pct = pca.explained_variance_ratio_[0] * 100
pc2_pct = pca.explained_variance_ratio_[1] * 100
print(f'PCA explained variance: PC1={pc1_pct:.1f}%, PC2={pc2_pct:.1f}%')

bands = bin_names_arr(scores_aa)

fig, ax = plt.subplots(figsize=(10, 8))
for b in BIN_NAMES:
    m = bands == b
    if m.sum() == 0: continue
    ax.scatter(diffs_2d[m, 0], diffs_2d[m, 1],
               s=12 if b != 'high' else 100,
               alpha=0.4 if b != 'high' else 1.0,
               color=colors_bin[b],
               edgecolors='black' if b == 'high' else 'none',
               linewidths=0.8,
               label=f'{b} (n={int(m.sum())})',
               zorder=3 if b == 'high' else 1)

ax.set_title(f'Pair-difference vectors |e_a - e_b| (PCA 2D) - K={VIZ_K}\n'
             f'Each point = one AA eval pair, colored by true normLev band')
ax.set_xlabel(f'PC1 ({pc1_pct:.1f}%)')
ax.set_ylabel(f'PC2 ({pc2_pct:.1f}%)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\ncolab15 (MSE training): PC1=8.7%, PC2=8.4%   (no visible band structure)')
print(f'colab16 (CE training):  PC1={pc1_pct:.1f}%, PC2={pc2_pct:.1f}%   '
      f'<-- compare here on rerun')